# Orion

## Getting Started

### Here we'll go through a complete rocket trajectory simulation for Orion, in the case of a Nominal Flight. 
#### Let's start by importing RocketPy

In [ ]:
%pip install rocketpy
from rocketpy import *
import numpy as np

## Setting Up the Simulation

#### To run the simulations locally complete the correct location for the csv files needed (part of the file path is already written)

### Environment

In [ ]:
env = Environment(latitude=-21.90795, longitude=-48.96156, elevation=495)

In [ ]:

env.set_date(
    (2026, 9, 4, 15)
)  # Hour given in UTC time

In [ ]:
env.set_atmospheric_model(
    type="custom_atmosphere",
    pressure=None,
    temperature=None, #Leaving temperature and pressure in None means we are getting values from ISA
    wind_u=2.77778,
    wind_v=0
    )

In [ ]:
env.all_info()

### Motor

In [ ]:
from rocketpy import Fluid, CylindricalTank, MassFlowRateBasedTank, HybridMotor

In [ ]:
import csv
#insert correct location for csv files to run the code locally.
liquid_mass=f'/Entrega RocketPy - PU4/Data/liquid_mass_curve_2026__455.csv'
vapor_mass=f'/Entrega RocketPy - PU4/Data/vapor_mass_curve_2026_455.csv'

flux_time = None

with open(liquid_mass, newline='', encoding="utf-8") as f:
    reader = csv.reader(f)

    for row in reader:
        if not row:          # skip empty lines
            continue
        value = row[0].strip()
        if value == "":      # skip empty first column
            continue
        flux_time = float(value)

print(f'Flux time: {flux_time}')

In [ ]:
# Define the fluids
oxidizer_liq = Fluid(name="N2O_l", density=742.9329783371916)
oxidizer_gas = Fluid(name="N2O_g", density=188.7650429545601)

# Define tank geometry
tank_shape = CylindricalTank(154 / 2000, 0.4034) 

oxidizer_tank = MassBasedTank(
    name = "TANK",
    geometry=tank_shape,
    flux_time=(flux_time), 
    liquid=oxidizer_liq,
    liquid_mass=liquid_mass,
    gas=oxidizer_gas,
    gas_mass=vapor_mass
)

In [ ]:
Nybrid = HybridMotor(
    thrust_source="/Entrega RocketPy - PU4/Data/Nemesis_Thrust_10_07_2026.csv",
    dry_mass=0,
    dry_inertia=(0, 0, 0),
    center_of_dry_mass_position=0.813,
    grain_number=1,
    grain_separation=0,
    grain_outer_radius= 98 / 2000,
    grain_initial_inner_radius= 50 / 2000,
    grain_initial_height= 300 / 1000,
    grain_density=900,
    nozzle_radius=63.36 / 2000,
    throat_radius=26 / 2000,
    interpolation_method="linear",
    grains_center_of_mass_position=357/1000,
    reshape_thrust_curve=False,
    nozzle_position=0,
    coordinate_system_orientation="nozzle_to_combustion_chamber"
)

In [ ]:
Nybrid.add_tank(
  tank = oxidizer_tank, position = (917.5+403.4/2)/1000
)

In [ ]:
Nybrid.all_info()

### Rocket

In [ ]:
Cd_RASAero = "/Entrega RocketPy - PU4/Data/CD_RasaeroII_Orion.csv"
Orion = Rocket(
    radius= 0.0817,
    mass=33.619, 
    inertia=(25.494, 25.495, 0.161), 
    power_off_drag= Cd_RASAero,
    power_on_drag= Cd_RASAero,
    center_of_mass_without_motor=2.079, 
    coordinate_system_orientation='nose_to_tail',
    )


rail_buttons = Orion.set_rail_buttons(
    upper_button_position=1.651,
    lower_button_position=3.371,
    angular_position=45,
)

In [ ]:
Orion.add_motor(Nybrid, position=3.393) 

#### Aerodynamic Surfaces

In [ ]:
naca0012 = "/Entrega RocketPy - PU4/Data/NACA0012-radians.csv"

nose_cone = Orion.add_nose(
      length=0.58763, kind="vonKarman", position=0
    )

fin_set = Orion.add_trapezoidal_fins(
    n=4,
    root_chord=0.135, 
    tip_chord=0.130, 
    span=0.125, 
    position=3.2691, 
    cant_angle=0.5,
    airfoil=(naca0012, "degrees"),
    sweep_length=0.005,
)

tail = Orion.add_tail(
    top_radius=0.0817, bottom_radius=0.06536, length=0.0518, position=3.4244
)

In [ ]:
fin_set.draw()

#### Parachutes

In [ ]:
Main = Orion.add_parachute(
    "Main",
    cd_s=5.54, 
    trigger=427, 
    sampling_rate=50,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

Drogue = Orion.add_parachute(
    "Drogue",
    cd_s=1.1, 
    trigger="apogee",
    sampling_rate=50,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

#### Rocket Info

In [ ]:
Orion.all_info()

## Simulating the Flight

In [ ]:
test_flight = Flight(
    rocket=Orion, environment=env, rail_length=8.2, inclination=80, heading=90
)

### Results

In [ ]:
test_flight.all_info()